# L69 · CTC（Connectionist Temporal Classification，CTC） 前向算法（forward algorithm） — 所有路径概率求和，O(T×2L) 纯 NumPy 实现

**学习目标**
- 理解 CTC 的核心挑战：不知道字符在哪一帧对齐（alignment）
- 掌握前向变量（forward variable，alpha） α 的递推定义
- 用纯 NumPy 实现 CTC 前向算法并数值验证

## CTC 的核心问题

声学模型输出 T 帧，标签序列有 L 个字符。帧数 T >> L，但不知道每个字符对应哪几帧。

CTC 解法：引入空白符 `<blank>`，允许所有合法的对齐路径，然后对它们的概率求和。

**合法路径** = 任意长度为 T 的序列，折叠后（去掉空白、合并重复）等于目标标签。

**扩展标签（extended label）** l' = `<b> l₁ <b> l₂ <b> ... <b> lL <b>`，长度 S = 2L+1。

**前向变量** α[t][s] = 在时刻 t，所有到达扩展标签位置 s 的路径前缀概率之和（log 域）。

**递推（三个合法前驱）**：
```
α[t][s] = ( α[t-1][s]              # 停留在 s
          + α[t-1][s-1]            # 从 s-1 跳来
          + α[t-1][s-2]  {仅当 l'[s] ≠ l'[s-2]}  # 跳过中间 blank
          ) × p[t][l'[s]]
```

**Log 域等价形式**（骨架代码使用此形式，避免浮点下溢）：
```
log α[t][s] = logsumexp(log α[t-1][s], log α[t-1][s-1], log α[t-1][s-2])
            + log p[t][l'[s]]
```
其中 `logsumexp(a, b) = np.logaddexp(a, b)`，将概率域的乘法转换为 log 域的加法，数值上安全。
具体说：`log(eᵃ + eᵇ) = np.logaddexp(a, b)`，而 `log(p × q) = log p + log q`。

In [ ]:
import numpy as np

In [ ]:
# 构造玩具示例：6 帧，词汇 {a:0, b:1, blank:2}，目标 'ab'
BLANK = 2
VOCAB_SIZE = 3
T = 6
LABELS = [0, 1]   # 'a', 'b'

np.random.seed(42)
logits = np.random.randn(T, VOCAB_SIZE)
# 转为 log 概率
# 数值稳定 log-softmax：先减最大值，避免 exp 溢出
logits_s = logits - logits.max(axis=1, keepdims=True)
log_probs = logits_s - np.log(np.exp(logits_s).sum(axis=1, keepdims=True))
print(f'log_probs shape: {log_probs.shape}  (T={T}, vocab={VOCAB_SIZE})')

## ✏️ 实现 CTC 前向算法

In [ ]:
def ctc_forward(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """CTC 前向算法：计算 log P(labels | log_probs)。

    Args:
        log_probs: (T, vocab_size) log-probability matrix.
        labels:    list of integer label ids (without blanks).
        blank:     blank token index.
                   注意：本笔记本 BLANK=2，请显式传入 blank=BLANK；
                   函数默认值 0 是 CTC 论文惯例，不适用于本例。

    Returns:
        log probability of the label sequence under CTC.
    """
    T, V = log_probs.shape
    L = len(labels)

    # 扩展标签：blank + label + blank + label + ... + blank
    lprime = [blank]
    for c in labels:
        lprime.append(c)
        lprime.append(blank)
    S = len(lprime)   # = 2L + 1

    NEG_INF = -1e30
    alpha = np.full((T, S), NEG_INF)

    # ✏️ TODO 步骤1：初始条件
    # 第 0 帧只能从扩展标签的前两个位置出发（blank 或第一个字符）
    # alpha[0, 0] = log_probs[0, lprime[0]]
    # if S > 1: alpha[0, 1] = log_probs[0, lprime[1]]

    # ✏️ TODO 步骤2：t=1..T-1 的递推（三个合法前驱）
    # for t in range(1, T):
    #     for s in range(S):
    #         val = alpha[t-1, s]                                   # 停留
    #         if s > 0:
    #             val = np.logaddexp(val, alpha[t-1, s-1])          # 跳1
    #         if s > 1 and lprime[s] != lprime[s-2]:                # 跳2（仅字符位）
    #             val = np.logaddexp(val, alpha[t-1, s-2])
    #         alpha[t, s] = val + log_probs[t, lprime[s]]

    # ✏️ TODO 步骤3：返回终态 log-sum
    # 最后一帧可停在位置 S-1（字符）或 S-2（blank），取 logaddexp
    # return np.logaddexp(alpha[-1, -1], alpha[-1, -2])

    return NEG_INF  # 占位符：完成步骤1-3后替换此行

In [ ]:
def ctc_forward_brute_force(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """暴力枚举版 CTC（仅用于验证，T 较小时可用）。
    枚举所有长度为 T 的路径，保留折叠后等于 labels 的路径，用 logsumexp 聚合。
    """
    from itertools import product
    T, V = log_probs.shape

    def collapse(path):
        """去掉 blank、合并重复字符 → 等价于 CTC collapse。"""
        result, prev = [], None
        for c in path:
            if c == blank:
                prev = None
            elif c != prev:
                result.append(c)
                prev = c
        return result

    log_p_list = []
    for path in product(range(V), repeat=T):
        if collapse(list(path)) == list(labels):
            lp = sum(log_probs[t, path[t]] for t in range(T))
            log_p_list.append(lp)

    if not log_p_list:
        return -1e30

    arr = np.array(log_p_list)
    m = arr.max()
    return float(m + np.log(np.exp(arr - m).sum()))


# 快速自检（T=4 时暴力枚举可在 < 1 秒内完成）
np.random.seed(42)
_lp4 = np.random.randn(4, VOCAB_SIZE)
_lp4_s = _lp4 - _lp4.max(axis=1, keepdims=True)
_lp4 = _lp4_s - np.log(np.exp(_lp4_s).sum(axis=1, keepdims=True))
_ref = ctc_forward_brute_force(_lp4, LABELS, blank=BLANK)
print(f"暴力枚举参考值（T=4）: {_ref:.4f}  → 用于断言 DP 实现正确性")

In [ ]:
try:
    lp = ctc_forward(log_probs, LABELS, blank=BLANK)
    print(f'log P(labels | frames) = {lp:.4f}')
    print(f'P(labels | frames)     = {np.exp(lp):.8f}')
    assert np.isfinite(lp), '前向算法应返回有限值'
    assert lp < 0, 'log 概率应为负数'

    # 强验证：与暴力枚举参考实现对照（T=6 时 3^6=729 路径，可在 < 1 秒内枚举）
    lp_ref = ctc_forward_brute_force(log_probs, LABELS, blank=BLANK)
    print(f'暴力枚举参考值         = {lp_ref:.4f}')
    assert np.isclose(lp, lp_ref, atol=1e-5), \
        f'❌ 与参考值不符：DP={lp:.4f}, brute={lp_ref:.4f}，请检查递推或初始条件'

    print('✅ CTC 前向算法验证通过（与暴力枚举吻合至 1e-5）')
except NotImplementedError:
    print('⚠️ ctc_forward 尚未实现，请完成 TODO 步骤1-3')

## 复杂度与直觉

In [ ]:
try:
    # 展示：T 增大时计算量线性增长（而不是指数增长）
    print(f'{"T":>4}  {"S=2L+1":>6}  {"计算格数":>8}  {"log P":>8}')
    print('-' * 35)
    for T_test in [5, 10, 20, 50, 100]:
        _logits_t = np.random.randn(T_test, VOCAB_SIZE)
        _logits_t -= _logits_t.max(axis=1, keepdims=True)  # 数值稳定
        lp_test = _logits_t - np.log(np.exp(_logits_t).sum(axis=1, keepdims=True))
        result = ctc_forward(lp_test, LABELS, blank=BLANK)
        S = 2*len(LABELS)+1
        print(f'{T_test:>4}  {S:>6}  {T_test*S:>8}  {result:>8.3f}')
except NotImplementedError:
    print('⚠️ ctc_forward 尚未实现，请先完成 TODO 步骤1-3')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| 扩展标签 l' | blank 插在每个标签之间，长度 2L+1 |
| 前向变量 α[t][s] | log 域递推，避免下溢 |
| 三个前驱 | 停留、跳 1、跳 2（限字符不同时）|
| 复杂度 | O(T × 2L) — 与暴力枚举路径的指数复杂度相比快得多 |
| L70 | Whisper 用 Attention 解码，但仍然是 token-by-token 的自回归过程 |

下一课（L70）从代码层面解剖 Whisper 架构：音频编码器（encoder）、交叉注意力（cross-attention）和文本解码器的完整实现。

## ✏️ 闭卷推导检查格 — CTC 前向算法 α 递推

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目 1 — α 递推公式**：CTC 前向变量 $\alpha(t, s)$ 满足递推：

$$\alpha(t, s) = \Big[\alpha(t-1, s) + \alpha(t-1, s-1) + \alpha(t-1, s-2) \cdot \mathbf{1}[\text{条件}]\Big] \cdot p(l'_s \mid x_t)$$

写出 $\mathbf{1}[\text{条件}]$ 的完整条件（涉及 blank 与相邻非 blank 标签的关系）。

**题目 2 — blank 合并规则**：

| 路径 | 折叠后标签 |
|------|-----------| 
| a - a | |
| a - ε - a | |
| a - ε - b | |

（ε = blank 符号）

**题目 3 — 扩展标签**：标签序列 $l = [a, b]$，写出 $l'$（含 blank）= ?

（在此处写推导...）

In [ ]:
# 验证：与预计算参考值做数值一致性对比（固定 seed=42）
import numpy as np

# 小规模例子：vocab={blank=0, a=1, b=2}，标签 l=[a,b]
# 扩展标签 l' = [ε, a, ε, b, ε]
T, vocab_size = 5, 3
np.random.seed(42)
log_probs_raw = np.random.dirichlet(np.ones(vocab_size), size=T)
log_probs = np.log(log_probs_raw + 1e-9)  # (T, vocab)

l_prime = [0, 1, 0, 2, 0]  # [blank, a, blank, b, blank]
S = len(l_prime)

# 前向递推（参考实现，不可修改）
NEG_INF = float('-inf')
log_alpha = np.full((T, S), NEG_INF)
log_alpha[0][0] = log_probs[0][l_prime[0]]
if S > 1: log_alpha[0][1] = log_probs[0][l_prime[1]]

for t in range(1, T):
    for s in range(S):
        paths = [log_alpha[t-1][s]]
        if s > 0:
            paths.append(log_alpha[t-1][s-1])
        # 第三路径：只在当前不是 blank 且与前一个非 blank 标签不同时才合法
        if s > 1 and l_prime[s] != 0 and l_prime[s] != l_prime[s-2]:
            paths.append(log_alpha[t-1][s-2])
        log_alpha[t][s] = np.logaddexp.reduce(paths) + log_probs[t][l_prime[s]]

# 最终 log P("ab" | 输入) = logaddexp(α(T-1, S-1), α(T-1, S-2))
log_p = np.logaddexp(log_alpha[-1][-1], log_alpha[-1][-2])

# 预计算参考值（seed=42 下固定）
REF_LOG_P = -1.361933   # 通过参考实现预先计算

assert not np.isnan(log_p), "log_p 是 NaN，递推可能有数值问题"
assert log_p < 0, "log 概率必须 < 0"
assert abs(log_p - REF_LOG_P) < 1e-4, (
    f"log_p = {log_p:.6f}，参考值 = {REF_LOG_P:.6f}\n"
    f"差值 = {abs(log_p - REF_LOG_P):.2e}（应 < 1e-4）\n"
    f"请检查题目 1 中 α 递推的三路径条件是否正确"
)
print(f"log P(ab | 输入) = {log_p:.6f}  参考值 = {REF_LOG_P:.6f}")
print("✅ CTC 前向算法验证通过（数值与参考一致）")